# 16.C21A/18.C21A Problem Set #4
Due Friday 1 May 2026 at 11:59 pm EDT

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Problem 0: Finite elements for 1-D diffusion with variable conductivity

If you didn't already do it, please solve Problem 3 from Problem Set #3 and turn it in with this problem set.


## Problem 1: Finite elements for diffusion across a material interface (10+6+8+14+6+4+12)

Consider a steady-state heat diffusion problem for two abutting materials with different thermal conductivies $k^a$ and $k^b$. The governing equations are
$$
(k^{a}T^{a}_{x})_x = -f^{a}, \qquad x\in(0,\tfrac{1}{2}),
$$
$$
(k^{b}T^{b}_{x})_x = -f^{b}, \qquad x\in(\tfrac{1}{2}, 1),
$$
with homogeneous Dirichlet boundary conditions
$$
T^{a}(x=0) = 0, \qquad T^{b}(x=1) = 0,
$$
and interface conditions to impose continuity of the solution (i.e., the temperature) and the heat flux:
$$
T^{a}(x=\tfrac{1}{2}) = T^{b}(x=\tfrac{1}{2}), \qquad
k^{a}(x=\tfrac{1}{2})\,T_x^{a}(x=\tfrac{1}{2}) =
k^{b}(x=\tfrac{1}{2})\,T_x^{b}(x=\tfrac{1}{2}).
$$

**1(a)** Using the Galerkin method with generic basis functions $\phi_i$, and ignoring boundary and interface conditions for this part only, write out the weighted residuals in a way that involves at most first derivatives. _(Hint: split your integrals into two parts, one for each material/subdomain.)_

**1(b)** Let the basis functions $\phi_i$ be a linear nodal basis and suppose that $N=6$ elements, all of equal size $\Delta x$, are used. Draw all the basis functions and label them. Also clearly label the element and node indices (i.e., things like $\mathrm{elem}_4$ and $x_3$). Accounting for the boundary conditions, how many degrees of freedom do you need to solve for?

**1(c)** Now apply the boundary and interface conditions to the result in part (a). What are the resulting weighted residual equations?

**1(d)** Continue to assume that all elements are of equal size. For the following scenarios, what are the entries of the stiffness matrix $\mathbf{K}$? (Treat $k^{a}$ and $k^{b}$ as known constants.)

  **(i)** $N=6$ elements.

  **(ii)** $N=5$ elements. _(Here, pay careful attention to the middle element.)_

**1(e)** Finally, let $f^{a}=f^{b}=1$. What is the system of equations to be solved? What are the entries of the right-hand-side "load vector" $\mathbf{f}$? (You do not need to rewrite the entries of the stiffness matrix; just refer to it as $\mathbf{K}$.)


**1(f)** **(coding)** Implement the FEM solution to this two-material problem in Python (or Julia) using the starter code below.

1. Complete the function `assemble_two_material(N, ka, kb)` so that it builds the global stiffness matrix $\mathbf{K}$ and load vector $\mathbf{f}$ for $N$ equal-sized elements on $[0,1]$, with $k^a$ on $[0,\tfrac12]$, $k^b$ on $[\tfrac12,1]$, $f^a=f^b=1$, and zero Dirichlet BCs. Handle elements that straddle the interface correctly.
2. Fix $(k^a,k^b)=(2,5)$ and run the simulation for $N=6$ and for $N=5$. Print both stiffness matrices and **verify numerically** that they agree with the closed-form expressions derived in 1(d).
3. Solve $\mathbf{K}\mathbf{T}=\mathbf{f}$ and plot both solutions on the same axes together with the exact solution
$$
T(x) = \begin{cases}
-\dfrac{x^2}{2k^a} + C_1 x, & 0 \le x \le \tfrac12,\\[4pt]
\dfrac{1 - x^2}{2k^b} + C_2 (x - 1), & \tfrac12 \le x \le 1,
\end{cases}
$$
with $C_1,C_2$ fixed by continuity of $T$ and of $k\,T_x$ at $x=\tfrac12$. Comment on whether $T$ is continuous and whether $T_x$ is continuous at the interface.

In [ ]:
# Problem 1(f): two-material FEM assembly (Python)

def assemble_two_material(N, ka, kb):
    """Assemble global stiffness and load for N equal elements on [0,1],
       k^a in [0,1/2] and k^b in [1/2,1], with homogeneous Dirichlet BCs
       and f^a = f^b = 1."""
    h = 1.0 / N
    nodes = np.linspace(0, 1, N + 1)
    K_full = np.zeros((N + 1, N + 1))
    F_full = np.zeros(N + 1)

    for e in range(N):
        a, b = nodes[e], nodes[e + 1]
        # TODO: determine k_int = integral of k(x) over this element
        #       (handle the element that straddles x = 1/2)
        # TODO: assemble element stiffness ke = (k_int / h**2) * [[1,-1],[-1,1]]
        # TODO: assemble element load fe = (h/2) * [1, 1]
        pass

    K = K_full[1:-1, 1:-1]
    F = F_full[1:-1]
    return K, F, nodes

# TODO: run for N = 6 and N = 5 with ka = 2, kb = 5
# TODO: verify K numerically against the analytical matrices from 1(d)
# TODO: solve and plot both FEM solutions together with the analytical solution

In [ ]:
# Problem 1(f): two-material FEM assembly (Julia)
using LinearAlgebra, Plots

function assemble_two_material(N, ka, kb)
    h = 1.0 / N
    nodes = range(0, 1, length=N + 1) |> collect
    K_full = zeros(N + 1, N + 1)
    F_full = zeros(N + 1)

    for e in 1:N
        a, b = nodes[e], nodes[e + 1]
        # TODO: determine k_int = integral of k(x) over this element
        # TODO: assemble element stiffness and load vector
    end

    K = K_full[2:N, 2:N]
    F = F_full[2:N]
    return K, F, nodes
end

# TODO: run for N = 6 and N = 5 with ka = 2, kb = 5
# TODO: verify K numerically against the analytical matrices from 1(d)
# TODO: solve and plot both FEM solutions together with the analytical solution

## Problem 2: Relationship between finite element and finite difference discretizations (12+10+14+6+10)

In this question, we will consider the relationship between Galerkin, *discontinuous Galerkin,* and upwind finite difference discretizations for the steady 1-D convection equation with a source term,
$$
u\frac{\partial T}{\partial x} = f(x).
$$
We will make the following assumptions:
- The velocity $u$ is constant in $x$ and is greater than zero, $u>0$.
- The source term $f(x)$ is a *linear* function of $x$.
- The grid spacing $\Delta x$ is constant. The number of elements in the domain is $N$. The domain begins at $x=0$ and ends at $x=L$.

For comparison to the finite element discretizations, recall the following three finite difference discretizations:
$$
\textbf{Central difference:} \quad u\frac{T_{i+1}-T_{i-1}}{2\Delta x} = f_i,
$$
$$
\textbf{1st-order upwind difference:} \quad u\frac{T_{i}-T_{i-1}}{\Delta x} = f_i,
$$
$$
\textbf{2nd-order upwind difference:} \quad u\frac{3 T_{i}-4T_{i-1} + T_{i-2}}{2\Delta x} = f_i.
$$

**2(a)** Continuous Galerkin with linear elements

Consider a standard Galerkin finite element discretization. Integrating the weighted residual by parts, show that the resulting equation for a *generic weighting function* $w(x)$ is,
$$
\left[wuT\right]^{L}_{0} - \int_{0}^{L} \frac{\partial w}{\partial x}\, u T\, dx = \int_{0}^{L} w f\,dx.
$$

Next, consider the specific case of *linear elements using a nodal basis* and apply a Galerkin method. For a weighting function $w = \phi_i$ corresponding to a node $i$ which is not at the domain boundary, show that the resulting discretization is
$$
\frac{u\left(T_{i+1}-T_{i-1}\right)}{2} = \frac{1}{6}\left[f(x_{i-1}) + 4f(x_i) + f(x_{i+1})\right]\Delta x,
$$
where $T_{i}$ are the approximations to the values of $T$ at node $i$, and $f_{i} = f(x_i)$.

_(Hint: to compute the integral $\int_{x_{i-1}}^{x_{i+1}} \phi_i(x) f(x) \, dx$, take advantage of the fact that $f(x)$ is linear and write it in terms of $f_{i-1}$, $f_i$, and $f_{i+1}$. )_

**2(b)** Discontinuous Galerkin with constant elements

Now we will consider a new type of finite element method known as a *discontinuous Galerkin method*. In this approach, the solutions and the weight functions are not required to be continuous from element to element. The simplest basis functions are constants in each cell. Specifically, the solution is assumed to be of the form,
$$
{T}(x) = \sum_{i=1}^{N} T_i \phi_i(x), \qquad
\phi_i = \begin{cases} 1 & \text{for } x_i < x < x_{i+1} \\ 0 & \text{otherwise.} \end{cases}
$$
In this case, $T_i$ represents the solution in element $i$. Using integration by parts, show that the weighted residual corresponding to the weighting function for element $i$ is,
$$
\left[u {T} \right]^{x_{i+1}}_{x_{i}} = \int^{x_{i+1}}_{x_{i}} f\,dx.
$$

The usual approach for the discontinuous Galerkin method is to use an *upwind flux* to evaluate $uT$ at the ends of the element (i.e., at $x = x_i$ and $x_{i+1}$). For example, $uT(x_i)$ would be evaluated using $T(x)$ from element $i-1$ since this element is upwind of node $i$. Using an upwind flux, and simplifying the source term since it has been assumed to be linear in $x$, show that the resulting discretization for this problem is,
$$
u T_{i} - u T_{i-1} = \frac{1}{2}\left[f(x_i) + f(x_{i+1})\right]\Delta x.
$$

**2(c)** Discontinuous Galerkin with linear elements

Now, consider the discontinuous Galerkin approximation with linear elements, i.e., the numerical solution varies linearly within each element. Instead of using a nodal basis within each element, use a basis for which the unknowns are the value of $T$ and its derivative $dT/dx$ at the midpoint of each element. For this basis, derive the weighted residual equations for each element. _(Note: there are two equations for each element)._

**2(d)** Comparisons with finite difference methods

How do the three finite element discretizations compare to the finite difference discretizations? Discuss both similarities and differences between the different discretizations.

**2(e)** **(coding)** Implement and compare the two schemes derived above on a concrete problem.

Take $u\,T_x = f(x)$ on $[0,L]$ with $u=1$, $L=1$, $f(x)=\sin(\pi x)$, and the inflow boundary condition $T(0)=0$. The exact solution is $T(x) = \dfrac{1-\cos(\pi x)}{\pi}$.

Note: $f$ is *no longer linear* in this part (e) of the problem, so quadratures for $\int_{0}^{L} \phi_i f\,dx$ might not be exact. Analogously to what you derived above, however, we will use Simpson's rule for continuous Galerkin and the trapezoidal rule for discontinuous Galerkin,  applied to the present nonlinear $f$.

1. Write a function that assembles and solves the **continuous Galerkin (linear)** discretization
$$
\frac{u(T_{i+1}-T_{i-1})}{2} = \frac{\Delta x}{6}\!\left[f(x_{i-1})+4f(x_i)+f(x_{i+1})\right],
$$
for $i=1,\ldots,N-1$, together with $T_0=0$ and a one-sided upwind closure at $i=N$: $u(T_N-T_{N-1})/\Delta x = \tfrac12[f(x_{N-1})+f(x_N)]$.
2. Write a function that assembles and solves the **discontinuous Galerkin (constant)** discretization
$$
u T_i - u T_{i-1} = \tfrac{1}{2}\!\left[f(x_i)+f(x_{i+1})\right]\Delta x,
$$
for $i=1,\ldots,N$, marched forward from $T_0 = 0$.
3. Run both schemes for $N \in \{10, 20, 40, 80, 160\}$. Plot the solutions for $N=20$ against the exact solution and make a log–log convergence plot of the discrete $L^2$ error (over the nodal values) vs $\Delta x$. Estimate the observed order of accuracy for each scheme and comment on how it compares to the classical FD analogues (central difference: $\mathcal{O}(\Delta x^2)$; first-order upwind: $\mathcal{O}(\Delta x)$).

In [2]:
# Problem 2(e): CG-linear vs DG-constant for u T_x = f on [0,1]
# u = 1, f(x) = sin(pi x), T(0) = 0, exact T(x) = (1 - cos(pi x)) / pi

def exact_T(x):
    return (1.0 - np.cos(np.pi * x)) / np.pi

def f_src(x):
    return np.sin(np.pi * x)

def solve_cg_linear(N, u=1.0, L=1.0):
    """Continuous Galerkin, linear nodal basis."""
    dx = L / N
    x = np.linspace(0, L, N + 1)
    # TODO: build the (N+1)x(N+1) system using the CG stencil and inflow BC T_0 = 0,
    #       with a one-sided upwind closure at i = N
    T = np.zeros(N + 1)
    return x, T

def solve_dg_constant(N, u=1.0, L=1.0):
    """Discontinuous Galerkin, constant elements, upwind flux."""
    dx = L / N
    x = np.linspace(0, L, N + 1)
    # TODO: march T_i from T_0 = 0 using u T_i - u T_{i-1} = (dx/2)[f(x_i)+f(x_{i+1})]
    T = np.zeros(N + 1)
    return x, T

# TODO: plot both for N = 20 against exact_T
# TODO: convergence study for N in {10,20,40,80,160}; log-log plot; estimate slopes

In [ ]:
# Problem 2(e): Julia version — CG-linear vs DG-constant
using LinearAlgebra, Plots

exact_T(x) = (1 - cos(π * x)) / π
f_src(x)   = sin(π * x)

function solve_cg_linear(N; u=1.0, L=1.0)
    dx = L / N
    x = range(0, L, length=N + 1) |> collect
    # TODO: build (N+1)x(N+1) system using CG stencil + inflow + upwind closure
    T = zeros(N + 1)
    return x, T
end

function solve_dg_constant(N; u=1.0, L=1.0)
    dx = L / N
    x = range(0, L, length=N + 1) |> collect
    # TODO: march T_i from T_0 = 0 using the DG upwind recurrence
    T = zeros(N + 1)
    return x, T
end

# TODO: plot for N = 20 against exact_T; convergence study log-log